# DistilVulSATD — one clean Colab run

**Run All (top → bottom) on Colab Pro / T4+.**

Pipeline:
1. Mount Drive + install deps
2. Load competition data
3. Distill official CodeBERT baseline → TinyBERT student
4. Save/overwrite checkpoint on Drive
5. Full public-test score (F1 + runtime + GFLOPs + submission_score)

Drive root: `MyDrive/NLBSE27_DistilVulSATD/`

In [ ]:
# Cell 1 — setup + Drive (overwrites old student outputs in the same folder)
!pip -q install "transformers>=4.36.0" "datasets>=2.16.0" "pandas" "numpy" "tqdm" "scikit-learn" "accelerate" "safetensors"

import json, re, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from google.colab import drive
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, TensorDataset
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/NLBSE27_DistilVulSATD')
RUN_NAME = 'tinybert_from_official_L256_T4_e3'  # fixed name => clean overwrite
OUT_DIR = DRIVE_ROOT / 'checkpoints' / 'student_runs' / RUN_NAME
BEST_DIR = OUT_DIR / 'best_model'
RESULTS_DIR = DRIVE_ROOT / 'results'

# Fresh folder for this clean run
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'logs').mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DRIVE_ROOT =', DRIVE_ROOT)
print('OUT_DIR    =', OUT_DIR)
print('device     =', device)
assert device.type == 'cuda', 'Enable GPU: Runtime → Change runtime type → GPU'

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/NLBSE27_DistilVulSATD
OUT_DIR    = /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/student_runs/tinybert_from_official_L256_T4_e3
device     = cuda


In [ ]:
# Cell 2 — knobs
TEACHER_NAME = 'NLBSE/nlbse27-code-comment-classification-baseline'
STUDENT_NAME = 'huawei-noah/TinyBERT_General_4L_312D'
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 32
LR = 3e-5
TEMPERATURE = 4.0
ALPHA = 0.7
VAL_FRACTION = 0.05
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print('knobs ready')

knobs ready


In [ ]:
# Cell 3 — helpers
def parse_label(v):
    if isinstance(v, (list, tuple, np.ndarray)):
        return [int(v[0]), int(v[1])]
    nums = re.findall(r'-?\d+', str(v))
    return [int(nums[0]), int(nums[1])]


def kd_loss(student_logits, teacher_logits, targets, temperature=4.0, alpha=0.7):
    # AMP-safe: BCE-with-logits only
    soft_t = torch.sigmoid(teacher_logits.float() / temperature).detach()
    soft = F.binary_cross_entropy_with_logits(
        student_logits.float() / temperature, soft_t
    ) * (temperature ** 2)
    hard = F.binary_cross_entropy_with_logits(
        student_logits.float(), targets.float()
    )
    return alpha * soft + (1.0 - alpha) * hard


def f1_table(preds, labels):
    rows = []
    for i, name in enumerate(['Vulnerability', 'MAT']):
        tp = int(((preds[:, i] == 1) & (labels[:, i] == 1)).sum())
        fp = int(((preds[:, i] == 1) & (labels[:, i] == 0)).sum())
        fn = int(((preds[:, i] == 0) & (labels[:, i] == 1)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({'cat': name, 'precision': p, 'recall': r, 'f1': f1})
    df = pd.DataFrame(rows)
    return df, float(df.f1.mean())


class FnDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(
            self.texts[i],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[i])
        return item


class DualTokDataset(Dataset):
    def __init__(self, texts, labels, teacher_tok, student_tok, max_length):
        self.texts = list(texts)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.teacher_tok = teacher_tok
        self.student_tok = student_tok
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        t = self.teacher_tok(
            self.texts[i], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        s = self.student_tok(
            self.texts[i], truncation=True, padding='max_length',
            max_length=self.max_length, return_tensors='pt',
        )
        return {
            't_input_ids': t['input_ids'].squeeze(0),
            't_attention_mask': t['attention_mask'].squeeze(0),
            's_input_ids': s['input_ids'].squeeze(0),
            's_attention_mask': s['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[i]),
        }

print('helpers ready')

helpers ready


In [ ]:
# Cell 4 — data
ds = load_dataset('NLBSE/nlbse27-code-comment-classification')
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df['label_list'] = train_df['Label'].apply(parse_label)
test_df['label_list'] = test_df['Label'].apply(parse_label)
train_df['vuln'] = train_df['label_list'].apply(lambda x: x[0])
train_df['satd'] = train_df['label_list'].apply(lambda x: x[1])

joint = train_df['vuln'].astype(str) + '_' + train_df['satd'].astype(str)
fit_df, val_df = train_test_split(
    train_df, test_size=VAL_FRACTION, random_state=SEED, stratify=joint
)
fit_df = fit_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

fit_texts = fit_df['Function'].tolist()
val_texts = val_df['Function'].tolist()
test_texts = test_df['Function'].tolist()
fit_labels = np.stack(fit_df['label_list'].to_numpy())
val_labels = np.stack(val_df['label_list'].to_numpy())
test_labels = np.stack(test_df['label_list'].to_numpy())

print(len(fit_df), 'fit |', len(val_df), 'val |', len(test_df), 'test')

README.md:   0%|          | 0.00/530 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 43.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.15MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/214586 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30655 [00:00<?, ? examples/s]

203856 fit | 10730 val | 30655 test


In [ ]:
# Cell 5 — teacher + student
teacher_tok = AutoTokenizer.from_pretrained(TEACHER_NAME)
student_tok = AutoTokenizer.from_pretrained(STUDENT_NAME)

teacher_cfg = AutoConfig.from_pretrained(
    TEACHER_NAME, num_labels=2, problem_type='multi_label_classification'
)
teacher = AutoModelForSequenceClassification.from_pretrained(TEACHER_NAME, config=teacher_cfg)
teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad = False

student_cfg = AutoConfig.from_pretrained(
    STUDENT_NAME,
    num_labels=2,
    problem_type='multi_label_classification',
    id2label={0: 'weakness', 1: 'MAT'},
    label2id={'weakness': 0, 'MAT': 1},
)
student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, config=student_cfg)
student.to(device)

print('teacher M params:', sum(p.numel() for p in teacher.parameters()) / 1e6)
print('student M params:', sum(p.numel() for p in student.parameters()) / 1e6)

config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

... (6 outputs left out)

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: huawei-noah/TinyBERT_General_4L_312D
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.bias            | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
fit_denses.{0, 1, 2, 3, 4}.weight          | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/arch

teacher M params: 124.64717
student M params: 14.350874


In [ ]:
# Cell 6 — distill train (saves best to Drive each improvement)
fit_loader = DataLoader(
    DualTokDataset(fit_texts, fit_labels, teacher_tok, student_tok, MAX_LENGTH),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR)
steps = max(len(fit_loader) * EPOCHS, 1)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler('cuda')


@torch.no_grad()
def eval_student(texts, labels, bs=64):
    student.eval()
    preds = []
    loader = DataLoader(FnDataset(texts, labels, student_tok, MAX_LENGTH), batch_size=bs)
    for batch in loader:
        batch.pop('labels')
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = student(**batch).logits
        preds.append((torch.sigmoid(logits) > 0.5).int().cpu().numpy())
    return f1_table(np.concatenate(preds, 0), np.asarray(labels))


history = []
best_f1 = -1.0

for epoch in range(1, EPOCHS + 1):
    student.train()
    tr_loss = 0.0
    for batch in tqdm(fit_loader, desc=f'distill e{epoch}'):
        y = batch['labels'].to(device)
        t_ids = batch['t_input_ids'].to(device)
        t_mask = batch['t_attention_mask'].to(device)
        s_ids = batch['s_input_ids'].to(device)
        s_mask = batch['s_attention_mask'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            with torch.no_grad():
                t_logits = teacher(input_ids=t_ids, attention_mask=t_mask).logits
            s_logits = student(input_ids=s_ids, attention_mask=s_mask).logits
            loss = kd_loss(s_logits, t_logits, y, TEMPERATURE, ALPHA)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tr_loss += float(loss.item())
    tr_loss /= max(len(fit_loader), 1)

    scores_df, avg_f1 = eval_student(val_texts, val_labels)
    print(f'epoch {epoch}: loss={tr_loss:.4f} val_avgF1={avg_f1:.4f}')
    print(scores_df.to_string(index=False))
    history.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'val_avg_f1': avg_f1,
        'per_class': scores_df.to_dict(orient='records'),
    })

    if avg_f1 > best_f1:
        best_f1 = avg_f1
        if BEST_DIR.exists():
            shutil.rmtree(BEST_DIR)
        BEST_DIR.mkdir(parents=True, exist_ok=True)
        student.save_pretrained(BEST_DIR)
        student_tok.save_pretrained(BEST_DIR)
        print('saved/overwrote best ->', BEST_DIR)

print('best_val_f1 =', best_f1)

distill e1:   0%|          | 0/6371 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 62.7MB            

model.safetensors: downloading bytes:           |  0.00B            

distill e1: 100%|██████████| 6371/6371 [29:28<00:00,  3.60it/s]


epoch 1: loss=5.7785 val_avgF1=0.5450
          cat  precision  recall       f1
Vulnerability   0.834991 0.69187 0.756723
          MAT   0.822222 0.20904 0.333333


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

distill e3: 100%|██████████| 6371/6371 [29:27<00:00,  3.60it/s]


epoch 3: loss=5.2415 val_avgF1=0.7437
          cat  precision   recall       f1
Vulnerability   0.853499 0.741899 0.793796
          MAT   0.858333 0.581921 0.693603


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved/overwrote best -> /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/student_runs/tinybert_from_official_L256_T4_e3/best_model
best_val_f1 = 0.7436991570203249


In [8]:
# Cell 7 — full public-test competition score + save to Drive
student = AutoModelForSequenceClassification.from_pretrained(BEST_DIR)
student_tok = AutoTokenizer.from_pretrained(BEST_DIR)
student.to(device).eval()

MAX_AVG_RUNTIME = 1000
MAX_AVG_FLOPS = 3_000_000
repetitions = 10
batch_size = 600

tokenized = student_tok(
    test_texts,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
)
loader = DataLoader(
    TensorDataset(tokenized['input_ids'], tokenized['attention_mask']),
    batch_size=batch_size,
)

preds = []
total_time = 0.0
total_flops = 0.0
pbar = tqdm(total=len(test_texts), desc='full score')
for batch in loader:
    input_ids, attention_mask = [x.to(device, non_blocking=True) for x in batch]
    begin = time.time()
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            with torch.profiler.profile(with_flops=True) as prof:
                for _ in range(repetitions):
                    outputs = student(input_ids=input_ids, attention_mask=attention_mask)
            total_flops += sum(k.flops for k in prof.key_averages()) / 1e9
    total_time += time.time() - begin
    probs = torch.sigmoid(outputs.logits)
    preds.extend((probs > 0.5).int().cpu().numpy().tolist())
    pbar.update(probs.size(0))
    del input_ids, attention_mask, outputs, probs
    torch.cuda.empty_cache()
pbar.close()

score_df, avg_f1 = f1_table(np.array(preds), test_labels)
avg_runtime = total_time / repetitions
avg_flops = total_flops / repetitions
submission = (
    0.6 * avg_f1
    + 0.2 * max((MAX_AVG_RUNTIME - avg_runtime) / MAX_AVG_RUNTIME, 0)
    + 0.2 * max((MAX_AVG_FLOPS - avg_flops) / MAX_AVG_FLOPS, 0)
)

payload = {
    'run_name': RUN_NAME,
    'teacher': TEACHER_NAME,
    'student': STUDENT_NAME,
    'max_length': MAX_LENGTH,
    'temperature': TEMPERATURE,
    'alpha': ALPHA,
    'epochs': EPOCHS,
    'best_val_f1': best_f1,
    'history': history,
    'checkpoint': str(BEST_DIR),
    'avg_f1': avg_f1,
    'avg_runtime': avg_runtime,
    'avg_flops': avg_flops,
    'submission_score': float(submission),
    'per_class': score_df.to_dict(orient='records'),
    'baseline_your_machine': 0.7165,
}

metrics_path = OUT_DIR / 'metrics.json'
results_path = RESULTS_DIR / 'student_tinybert_from_official_full_score.json'
with open(metrics_path, 'w') as f:
    json.dump(payload, f, indent=2)
with open(results_path, 'w') as f:
    json.dump(payload, f, indent=2)

print('----------------RESULTS--------------------')
print(score_df.to_string(index=False))
print('avg F1:', avg_f1)
print('avg runtime:', avg_runtime)
print('avg GFLOPs:', avg_flops)
print('submission_score:', round(submission, 4))
print('baseline was ~0.7165')
print('SAVED checkpoint:', BEST_DIR)
print('SAVED metrics:', metrics_path)
print('SAVED results:', results_path)

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

full score: 100%|██████████| 30655/30655 [08:07<00:00, 62.88it/s]

----------------RESULTS--------------------
          cat  precision   recall       f1
Vulnerability   0.866080 0.753382 0.805810
          MAT   0.925234 0.588119 0.719128
avg F1: 0.7624690380262247
avg runtime: 48.54978859424591
avg GFLOPs: 81535.36562038395
submission_score: 0.8423
baseline was ~0.7165
SAVED checkpoint: /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/student_runs/tinybert_from_official_L256_T4_e3/best_model
SAVED metrics: /content/drive/MyDrive/NLBSE27_DistilVulSATD/checkpoints/student_runs/tinybert_from_official_L256_T4_e3/metrics.json
SAVED results: /content/drive/MyDrive/NLBSE27_DistilVulSATD/results/student_tinybert_from_official_full_score.json
